# 8. Variability classes: what a star does on its own

A transit is a dip in the light of a star. Before a dip can be a planet, the star
itself must be understood, because many stars change their brightness on their own.
This notebook shows the kinds of change that matter to the benchmark, with one real
light curve from the cache for each kind, and it shows where the benchmark reads a
star's class and what it does with it.

The word "class" here means a label from the Gaia mission. Gaia measured the
brightness of about two billion stars some seventy times each between 2014 and 2025.
Its data-processing team sorted the stars whose brightness changed into named types
by the shape and the period of the change, and published the labels in the third
data release as the variability classification table [1]_, [2]_. The benchmark reads
that table through Vizier, the catalogue service of the Strasbourg data centre, and
reads the TESS eclipsing-binary catalogue [3]_ beside it.

A light curve in this notebook is the flux of one star in TESS sector 98, two-minute
cadence, as the pipeline delivered it, with time counted from the first point. The
sector has four gaps, where the spacecraft sent its data to Earth.

## Eclipsing binary (Gaia class `ECL`)

Two stars orbit each other. When one passes in front of the other, the light drops,
and the drop repeats every orbit. The drops are deep, often tens of percent, and they
come in two sizes, because the two stars are not the same: a deep dip when the
fainter star hides the brighter one, a shallow dip half an orbit later when the
brighter hides the fainter.

![Eclipsing binary, TIC 231822771](figures/class_eclipsing_binary.png)

TIC 231822771 drops by a third to a half every day, and its two dips are nearly the
same depth, so its two stars are alike. The TESS eclipsing-binary catalogue lists it
and Gaia classes it `ECL`. A planet transit is the same geometry with a planet in
place of the second star, which is why an eclipsing binary is the first impostor to
rule out: a binary seen at a grazing angle, or a binary blended with a brighter
neighbour, can make a dip as shallow as a planet's. The benchmark removes every star
in either catalogue from its target lists, and the tools `odd_even` and
`secondary_eclipse` look for the two dip sizes and the dip at half phase in whatever
is left.

## Rotating spotted star (Gaia class `RS`)

A cool star with large dark spots on its surface dims when the spots face us and
brightens when they turn away. The change is smooth, repeats with the star's rotation
period, days to weeks, and its shape drifts from one rotation to the next as spots
grow and fade. Such stars also flare: a sharp rise of a few percent that decays in
minutes to an hour.

![Rotating spotted star, TIC 21505902](figures/class_rotating_spotted.png)

TIC 21505902 varies by 8 percent every 5.8 days, and each cycle has a different
shape. The benchmark's filter removes the smooth variation before the search, but
the hour-scale noise that the spots and flares leave behind is several times the
white noise, and that is what hides a shallow planet on such a star (LOGBOOK
2026-10-31). The two injected planets the benchmark placed on this star are missed by
every model.

## Solar-like star (Gaia class `SOLAR_LIKE`)

A star like the Sun has small spots and a mild rotation signal of a fraction of a
percent, plus occasional small flares. Gaia gives the label to stars whose variation
has the amplitude and the timescale of the Sun's.

![Solar-like star, TIC 160330305](figures/class_solar_like.png)

TIC 160330305 moves by about one percent over the sector in slow waves. This is the
easy case for a transit search: the filter removes the waves, and a dip of a few
hundred parts per million stands clear of what remains.

## Pulsating star

Some stars swell and shrink on a fixed rhythm, from hours to days, and their
brightness follows. Gaia has several classes for these by period and shape: `RR`
for RR Lyrae stars, with periods under a day and a sharp rise then a slow decline;
`DSCT|GDOR|SXPHE` for the short-period pulsators of hours; `CEP` for Cepheids, with
periods of days; `MIRA|SR` for the slow red giants.

![TIC 55270064 and its 0.946-day sawtooth](figures/class_pulsator.png)

TIC 55270064 carries no Gaia class, and the benchmark found its signal itself: a
0.1-percent sawtooth every 0.946 days, a sharp rise then a slow fall, the shape of an
RR Lyrae star diluted to a tenth of a percent, either a low-amplitude pulsator or a
pulsating neighbour 7 arcseconds away (LOGBOOK 2026-10-25). The box search reads the
steep part of each cycle as a transit, and every vetting tool that compares the dip
with the median of the rest passes it. The tool `orbit_shape` was built for this
star: it judges the whole orbit, not the dip, and reads `variable` where the flux
rises and falls through the period.

## A planet host, for comparison

![Confirmed planet host, TIC 25155310](figures/class_planet_host.png)

TIC 25155310 is the host of TOI-114.01, a confirmed planet with a period of 3.29
days and a transit of 0.7 percent. The star itself is quiet, and the dips are short,
flat-bottomed and all the same depth. Folded at the catalogue period and epoch,
every dip lands at phase zero. This is the signal every tool in the benchmark is
built to confirm, and the light curve comes from the real-planet episode set
`s98_real_v1`, where the flux is as observed and the hidden answer is the catalogue
entry.

## Other Gaia classes a reader will meet

| class | what the star does | what it means for a transit search |
|---|---|---|
| `ECL` | two stars eclipse each other | the first impostor; excluded from the target lists |
| `RS` | spots rotate in and out of view | smooth waves of percent size, flares, red noise at the hour scale |
| `SOLAR_LIKE` | mild spots, like the Sun | the easy case |
| `RR`, `CEP`, `DSCT\|GDOR\|SXPHE`, `MIRA\|SR` | the star pulsates | a sawtooth or a sinusoid that the box search can read as a dip |
| `BE\|GCAS\|SDOR\|WR` | a hot star sheds gas in bursts | irregular changes, no period |
| `ACV\|CP\|MCP\|ROAM\|ROAP\|SXARI` | a chemically peculiar star with surface patches | smooth rotation signal like `RS` |
| `YSO` | a young star still accreting | large irregular dips that can look like a transit |
| `AGN`, `WD`, `MICROLENSING` | not a normal star at all | rare among TESS two-minute targets |

The labels and their definitions are from the Gaia DR3 variability classification
[1]_; the table above gives the reading the benchmark takes of each.

## Where the class enters the benchmark

The class is read twice. When the target list of a sector is drawn, every star in the
TESS eclipsing-binary catalogue or with the Gaia class `ECL` is removed, and the
removed stars are recorded in the targets file. When the context of an episode set is
fetched, the classes of each host are stored in `context.json` and the tool
`ephemeris_match` returns them to the model, so the model can read "this star is a
known rotator" before it decides what a 0.9-day signal is. The cell below shows the
classes of the hosts of the episode set `s98_v3`.

In [ ]:
import json
from collections import Counter
from pathlib import Path

context = json.loads(Path("../data/episodes/s98_v3/context.json").read_text())
classes = {record["tic"]: record["gaia_classes"] for record in context.values()}
print(Counter(tuple(value) for value in classes.values()))
print({tic: value for tic, value in classes.items() if value})

Most hosts carry no class at all: Gaia labels only the stars whose variation is
clear in its own sparse sampling, and a star that TESS shows as a mild rotator can
be unlabelled in Gaia. A missing class means "not labelled", never "constant".

The two catalogues come from Vizier alone. When the Strasbourg servers were down for
several days at the end of October 2026, the context of the real-planet set was
fetched with the Gaia sources from the copy at MAST and the classes and the
eclipsing-binary entries left empty, recorded as unavailable in the source record of
each context (LOGBOOK 2026-11-01). A context says where each field came from, so a
reader can tell an empty field from a star that was checked and found plain.

## References

.. [1] Rimoldini, L., et al. (2023). Gaia Data Release 3: All-sky classification of
   12.4 million variable sources into 25 classes. Astronomy and Astrophysics, 674,
   A14. https://doi.org/10.1051/0004-6361/202245591
.. [2] Eyer, L., et al. (2023). Gaia Data Release 3: Summary of the variability
   processing and analysis. Astronomy and Astrophysics, 674, A13.
   https://doi.org/10.1051/0004-6361/202244242
.. [3] Prša, A., et al. (2022). TESS Eclipsing Binary Stars. I. Short-cadence
   observations of 4584 eclipsing binaries in sectors 1 to 26. The Astrophysical
   Journal Supplement Series, 258, 16. https://doi.org/10.3847/1538-4365/ac324a